# Libraries and Utility Functions

In this problem, you will implement a particle filter for state estimation of the 7DOF Kuka iiwa robot, specifically for the first two joints only. We will assume that a range-and-bearing sensor is located at the end effector of the robot for our measurement model and that there are several beacons in the workspace.

In [0]:
# Run once at the beginning to install pybullet
!pip install pybullet

In [0]:
import pybullet as p
import pybullet_data
import numpy as np
from scipy.stats import norm
from unittest.mock import patch

import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

plt.rcParams["animation.html"] = "jshtml"
plt.rcParams['figure.dpi'] = 90
plt.ioff()

In [0]:
# Utility function for kinematics calculations

def get_ee_pos(q_vec):
    orig_states = [p.getJointState(robot_id, i)[0] for i in range(7)]

    p.resetJointState(robot_id, active_joints[0], q_vec[0])
    p.resetJointState(robot_id, active_joints[1], q_vec[1])

    ee_pos = np.array(p.getLinkState(robot_id, ee_link_idx, computeForwardKinematics=1)[0])

    for idx, pos in enumerate(orig_states):
        p.resetJointState(robot_id, idx, pos)
    p.getLinkState(robot_id, ee_link_idx, computeForwardKinematics=1)

    return ee_pos

In [0]:
# Static visualization function

def visualize(history, dt=0.05):
    time = np.arange(len(history)) * dt

    true_q = np.array([h['true_q'] for h in history])
    est_q = np.array([h['est_q'] for h in history])

    true_ee = np.array([h['true_ee'] for h in history])
    est_ee = np.array([h['est_ee'] for h in history])

    # Calculate tracking RMSE
    q0_rmse = np.sqrt(np.mean((true_q[:, 0] - est_q[:, 0])**2))
    q1_rmse = np.sqrt(np.mean((true_q[:, 1] - est_q[:, 1])**2))
    x_rmse  = np.sqrt(np.mean((true_ee[:, 0] - est_ee[:, 0])**2))
    y_rmse  = np.sqrt(np.mean((true_ee[:, 1] - est_ee[:, 1])**2))

    # Setup stacked subplots with shared time axis
    fig, (ax_q, ax_ee) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

    # --- Top Subplot: Joint Angles Tracking vs Time ---
    ax_q.plot(time, true_q[:, 0], 'g-', lw=2, label=f'$q_0$ True')
    ax_q.plot(time, est_q[:, 0], 'm--', lw=2, label=f'$q_0$ Est (RMSE: {q0_rmse:.3f} rad)')
    ax_q.plot(time, true_q[:, 1], 'b-', lw=2, label=f'$q_1$ True')
    ax_q.plot(time, est_q[:, 1], 'c--', lw=2, label=f'$q_1$ Est (RMSE: {q1_rmse:.3f} rad)')

    ax_q.set_ylabel("Joint Angle (rad)", fontsize=10)
    ax_q.set_title("Particle Filter Tracking Performance Over Time", fontsize=12, fontweight='bold')
    ax_q.grid(True, linestyle='--', alpha=0.5)
    ax_q.legend(loc='upper right', fontsize=9, framealpha=0.9)

    # --- Bottom Subplot: End-Effector Cartesian Trajectory vs Time ---
    ax_ee.plot(time, true_ee[:, 0], 'g-', lw=2, label=f'X True')
    ax_ee.plot(time, est_ee[:, 0], 'm--', lw=2, label=f'X Est (RMSE: {x_rmse:.3f} m)')
    ax_ee.plot(time, true_ee[:, 1], 'b-', lw=2, label=f'Y True')
    ax_ee.plot(time, est_ee[:, 1], 'c--', lw=2, label=f'Y Est (RMSE: {y_rmse:.3f} m)')

    ax_ee.set_xlabel("Time (seconds)", fontsize=10)
    ax_ee.set_ylabel("End-Effector Position (m)", fontsize=10)
    ax_ee.grid(True, linestyle='--', alpha=0.5)
    ax_ee.legend(loc='upper right', fontsize=9, framealpha=0.9)

    plt.tight_layout()
    plt.show()

In [0]:
# Animation function

def animate(frame):
    ax_joint.cla()
    ax_work.cla()

    # --- Subplot 1: Joint Space (q0 vs q1) ---
    ax_joint.set_xlim(-np.pi, np.pi)
    ax_joint.set_ylim(-np.pi, np.pi)
    ax_joint.grid(True, linestyle='--', alpha=0.5)
    ax_joint.set_title(f"Joint Space (q0 vs q1) | Step {frame}", fontsize=11)
    ax_joint.set_xlabel("Joint 0 Angle (rad)")
    ax_joint.set_ylabel("Joint 1 Angle (rad)")

    q_particles = history[frame]['q_particles']
    ax_joint.scatter(q_particles[:, 0], q_particles[:, 1], c='deepskyblue', alpha=0.3, s=15, label='Particles (q)')

    true_q_traj = np.array([history[i]['true_q'] for i in range(frame + 1)])
    est_q_traj = np.array([history[i]['est_q'] for i in range(frame + 1)])

    ax_joint.plot(true_q_traj[:, 0], true_q_traj[:, 1], 'g-', lw=2, label='True q')
    ax_joint.plot(est_q_traj[:, 0], est_q_traj[:, 1], 'm--', lw=2, label='Estimated q')
    ax_joint.plot(true_q_traj[-1, 0], true_q_traj[-1, 1], 'go', ms=7)
    ax_joint.plot(est_q_traj[-1, 0], est_q_traj[-1, 1], 'mo', ms=7)
    ax_joint.legend(loc='upper right', fontsize=8)

    # --- Subplot 2: Workspace (X vs Y) ---
    ax_work.set_xlim(-1.2, 1.2)
    ax_work.set_ylim(-1.2, 1.2)
    ax_work.set_aspect('equal')
    ax_work.grid(True, linestyle='--', alpha=0.5)
    ax_work.set_title(f"Workspace (X vs Y) | Step {frame}", fontsize=11)
    ax_work.set_xlabel("X Position (m)")
    ax_work.set_ylabel("Y Position (m)")

    ax_work.scatter(beacons[:, 0], beacons[:, 1], c='red', marker='^', s=100, label='Beacons', zorder=5)

    particles_ee = history[frame]['particles_ee']
    ax_work.scatter(particles_ee[:, 0], particles_ee[:, 1], c='deepskyblue', alpha=0.3, s=15, label='Particles (EE)')

    true_ee_traj = np.array([history[i]['true_ee'] for i in range(frame + 1)])
    est_ee_traj = np.array([history[i]['est_ee'] for i in range(frame + 1)])

    ax_work.plot(true_ee_traj[:, 0], true_ee_traj[:, 1], 'g-', lw=2, label='True EE')
    ax_work.plot(est_ee_traj[:, 0], est_ee_traj[:, 1], 'm--', lw=2, label='Estimated EE')
    ax_work.plot(true_ee_traj[-1, 0], true_ee_traj[-1, 1], 'go', ms=7)
    ax_work.plot(est_ee_traj[-1, 0], est_ee_traj[-1, 1], 'mo', ms=7)
    ax_work.legend(loc='upper right', fontsize=8)

    plt.tight_layout()

# Particle Filter

The `ParticleFilter2DOF` class implements a particle filter by initializing and updating a $M \times 2$ array, where $M$ is the number of particles and each particle is a $(q_0,q_1)$ tuple within joint limits. There is also an array of $M$ weights, one for each particle.

In `step()`, the key procedures are motion prediction, measurement update, and resampling, which you will implement as separate functions. The `estimate()` method computes the average of the particles to give us a point estimate of the robot's joint values.

`simulate_particle_filter()` executes an entire trajectory over time. It provides a sequence of commanded velocities to the robot, computes a sequence of noisy measurements, and runs the particle filter to perform estimation. A `history` of the filter is then returned.

In [0]:
class ParticleFilter2DOF:
    def __init__(self, num_particles=200, joint_limits=(-np.pi, np.pi)):
        self.num_particles = num_particles
        self.joint_limits = joint_limits
        self.particles = np.random.uniform(
            low=joint_limits[0], high=joint_limits[1], size=(num_particles, 2)
        )
        self.weights = np.ones(num_particles) / num_particles

    def step(self, control_u, z_measurements, dt, motion_noise, measure_noise_r, measure_noise_b):
        self.particles = predict_step(self.particles, control_u, self.joint_limits, dt, motion_noise)
        self.weights = update_step(self.particles, self.weights, z_measurements, measure_noise_r, measure_noise_b)
        self.particles, self.weights = resample_step(self.particles, self.weights)

    def estimate(self):
        return np.average(self.particles, weights=self.weights, axis=0)

In [0]:
def simulate_particle_filter(motion_noise, measure_noise_r, measure_noise_b, num_steps=200, dt=0.05):
    pf = ParticleFilter2DOF()
    history = []
    substeps = int(dt / (1.0 / 240.0))
    noise_r_std = 0.04
    noise_b_std = 0.05

    # Execution & Data Logging Loop
    for step in range(num_steps):
        # Sudden Disturbance / "Kidnapped Joint" (e.g., at Step 100)
        if step == 100:
            bump_joint_idx = active_joints[0]
            current_angle = p.getJointState(robot_id, bump_joint_idx)[0]
            p.resetJointState(robot_id, bump_joint_idx, current_angle - 0.5)

        t = step * dt
        target_velocities = np.array([0.8 * np.sin(1.2 * t), 0.6 * np.cos(0.8 * t)])
        noisy_target_velocities = target_velocities + np.random.normal(0, 0.03, size=len(target_velocities))

        for idx, j_idx in enumerate(active_joints):
            p.setJointMotorControl2(
                robot_id, j_idx, p.VELOCITY_CONTROL,
                targetVelocity=noisy_target_velocities[idx], force=200
            )

        for _ in range(substeps):
            p.stepSimulation()

        q_true = np.array([p.getJointState(robot_id, j)[0] for j in active_joints])
        true_ee_pos = np.array(p.getLinkState(robot_id, ee_link_idx, computeForwardKinematics=1)[0])

        dx = beacons[:, 0] - true_ee_pos[0]
        dy = beacons[:, 1] - true_ee_pos[1]

        true_ranges = np.sqrt(dx**2 + dy**2)
        true_bearings = np.arctan2(dy, dx)

        z_ranges_noisy = true_ranges + np.random.normal(0, noise_r_std, size=len(beacons))
        raw_noisy_bearings = true_bearings + np.random.normal(0, noise_b_std, size=len(beacons))
        z_bearings_noisy = np.arctan2(np.sin(raw_noisy_bearings), np.cos(raw_noisy_bearings))
        z_noisy = np.column_stack([z_ranges_noisy, z_bearings_noisy])

        pf.step(target_velocities, z_noisy, dt, motion_noise, measure_noise_r, measure_noise_b)
        q_est = pf.estimate()
        est_ee_pos = get_ee_pos(q_est)
        particle_ee_positions = np.array([get_ee_pos(p_q)[:2] for p_q in pf.particles])

        # Save state logs for both spaces
        history.append({
            'true_q': q_true.copy(),
            'est_q': q_est.copy(),
            'q_particles': pf.particles.copy(),
            'true_ee': true_ee_pos[:2],
            'est_ee': est_ee_pos[:2],
            'particles_ee': particle_ee_positions
        })

    return history

# 4.1: Motion Prediction (3 pts)

We will use a simple velocity control model with additive Gaussian noise. Given the current particle configurations $\mathbf q$ and a control velocity $\mathbf u$, first deterministically update each particle using $\mathbf q \leftarrow \mathbf q + \mathbf u dt$. Then add independent Gaussian noise to both components of every particle sampled from $\mathcal N(0, \sigma^2)$, where $\sigma$ is the provided standard deviation.

Finally, clip all joint angles to the joint limits. All configurations less than the min should be set to the min limit, and similarly for the max.


In [0]:
def predict_step(particles, control_u, joint_limits, dt, std_noise):
    """
    Predicts next particle positions given velocity u.
    Returns: Updated particles array.
    """
    # TODO
    return particles

# 4.2: Measurement Update (12 pts)

In `update_step()`, the goal is to compute and return the set of updated particle weights due to measurements. Given that there are $B$ beacons in the workspace, `z_measurements` is a $B \times 2$ array containing range and bearing measurements from the end effector to each beacon. The bearings are made with respect to the inertial $x$ axis, so we will not worry about end effector orientation.

Iterate over each particle and compute the corresponding end effector position. Compute the expected range and bearing to each beacon, followed by the *difference* between the given and expected measurements $\Delta \mathbf r$ and bearings $\Delta \mathbf \phi$. For the latter, remember to convert to a standard angular distance, *i.e.*, $\min(|\Delta\phi|, 2\pi-|\Delta\phi|)$.

Next, we need the observation likelihood for each particle. We will assume independent range and bearing measurements for each beacon, so the overall likelihood for each particle is the cumulative product of the likelihoods over *both* range and bearing for *all* beacons. The individual range and bearing likelihoods can be found using `scipy.stats.norm.pdf()`, where the center (`loc`) is $0$, the `scale` is the corresponding standard deviation, and the evaluation is done at $\Delta \mathbf r$ or $\Delta \mathbf \phi$.

Finally, multiply the cumulative likelihoods by the `weights` that were passed in (they may not be $1$ as we may be resampling in each step), and return the updated weights.

In [0]:
def update_step(particles, weights, z_measurements, noise_std_r, noise_std_b):
    """
    Computes likelihoods and updates particle weights.
    Returns: Updated weights array.
    """
    # TODO
    return weights

Since this method is quite involved, you should verify its correctness using the unit test below before proceeding.

In [0]:
def test_update_step():
    # 1. Define particles directly as mock EE positions to avoid dict/lookup hacks
    particles = np.array([[0.0, 0.0], [0.0, 0.5]])
    weights = np.array([0.6, 0.4])
    expected_weights = np.array([1.55068e2, 3.52228e-7])

    z_measurements = np.array([[1.0, 0.0], [1.00005, -3.15159]])

    # 2. Temporarily patch globals for the test scope (automatically cleans up on exit)
    test_mocks = {
        "beacons": np.array([[1.0, 0.0], [-1.0, 0.01]]),
        "get_ee_pos": lambda p: p,  # Identity function
    }

    with patch.dict(globals(), test_mocks):
        updated_weights = update_step(
            particles,
            weights,
            z_measurements,
            noise_std_r=0.09,
            noise_std_b=0.11,
        )

    assert np.allclose(updated_weights, expected_weights, rtol=1e-4), (
        f"Weight mismatch!\nGot: {updated_weights}\nExpected: {expected_weights}"
    )

    print("✓ update_step unit test passed successfully!")


# Execute unit test
test_update_step()

# 4.3: Resampling (6 pts)

The last subroutine is resampling. First, normalize the weights by dividing them by their sum. Then resample the particles if the *effective sample size* is less than a threshold:

$$N_{eff} = \frac1{\sum_m w_i^2}< \frac{M}{2}$$

This is a simple way to estimate variance. If all particles contribute equally, then $N_{eff} = M$; if only one particle has nonzero weight (extreme degeneracy), then $N_{eff} = 1$. If resampling is needed, simply use `np.random.choice()` to sample $M$ new particles from the weighted distribution, and reset the weights to $1$.

If resampling is not needed, do not modify the particles or weights. In both cases, return the particles and weights when done.

In [0]:
def resample_step(particles, weights):
    """
    Normalizes weights, and then resamples particles if variance is sufficiently high.
    Returns: (new_particles, reset_weights).
    """
    # TODO
    return particles, weights

# 4.4: Experimentation (4 pts)

We can now test the particle filter. First run the cell below with the code provided. You should see that the filter tracks the robot reasonably well, with plots showing both the C-space and workspace progression. There is a disturbance to one of the joints at 5 seconds, but the filter is able to recover very quickly.

Experiment with separately setting low and high values of the motion noise and measure noise parameters. For example, set motion noise to $0$, and then to $0.3$. Then set it back to $0.03$, and set measurement noise (both range and bearing) to $0.01$, and then to $1.0$.

For each setting, briefly compare your observations with that of the baseline result. Also give a reason **explaining** why the filter performs differently for each. For submission, run and leave the output of the setting with the original parameter values of $0.03, 0.04, 0.05$.

[Describe your observations here.]

In [0]:
np.random.seed(100)

sim_id = p.connect(p.DIRECT)
p.setAdditionalSearchPath(pybullet_data.getDataPath())

robot_id = p.loadURDF("kuka_iiwa/model.urdf", useFixedBase=True, physicsClientId=sim_id)
p.setGravity(0, 0, -9.81, physicsClientId=sim_id)
p.setRealTimeSimulation(1, physicsClientId=sim_id)

active_joints = [0, 1]  # Lock joints 2-6 to keep state space 2D
ee_link_idx = 6

# Fixed workspace beacons (x, y, z)
beacons = np.array([
    [0.6,  0.6, 0.5],
    [-0.6, 0.6, 0.5],
    [0.0, -0.6, 0.5]
])

motion_noise = 0.03
measure_noise_r = 0.04
measure_noise_b = 0.05
history = simulate_particle_filter(motion_noise, measure_noise_r, measure_noise_b)
p.disconnect()

visualize(history)

You can also view an animation of the particle filter in both the C-space and the workspace. This should give you a clearer idea of the estimated joint and end effector trajectories over time.

In [0]:
fig, (ax_joint, ax_work) = plt.subplots(1, 2, figsize=(13, 6))
anim = animation.FuncAnimation(fig, animate, frames=len(history), interval=60)
plt.close(fig)

HTML(anim.to_jshtml())